# Clustering Songs by Sound

**Goal:** group songs into clusters by **how they sound**, using only Spotify's audio features, in a way that works on **any music dataset**, not just the one it was built on. A future recommender can then use the clusters and features to find songs that sound similar to a chosen song.

**Steps**

1. Load and clean the data
2. Prepare the audio features on a fixed scale
3. Choose the number of clusters
4. Cluster the songs and describe each cluster
5. Save the results for a future recommender

## Settings

All the choices you might want to change are here, so the rest of the notebook can stay as it is.

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Hide a harmless scikit-learn warning that k-means prints on Windows
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak")

# Where the exports are, and where results are saved
# (works whether the notebook runs from the project root or this subfolder)
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
OUTPUT_DIR = DATA_DIR / "processed"

# Audio features used to describe each song's sound (Step 2 explains the choice)
FEATURES = ["Danceability", "Energy", "Loudness", "Speechiness",
            "Acousticness", "Instrumentalness", "Valence"]

# Fixed feature preparation (Step 2): the same for every dataset, nothing is learned from the data
LOUDNESS_RANGE = (-60, 0)  # Spotify's loudness range in dB, mapped to 0-1
SQRT_FEATURES = ["Speechiness", "Acousticness", "Instrumentalness"]  # bunched near 0, so spread out with a square root

# Clustering (Steps 3-4)
K_OPTIONS = range(2, 13)   # numbers of clusters compared in Step 3
N_CLUSTERS = 6             # number of clusters used, chosen in Step 3
STABILITY_REPEATS = 10     # random half-splits per k in the stability test
RANDOM_STATE = 42          # makes results repeatable

## Step 1 — Load and clean the data

**What was done:** every export in `data/` was combined into one table, and only rows that are real, analyzable songs were kept, one row per song.

**How:**

- **Loading.** Every CSV directly inside `data/` is read. Exports can be saved with different text encodings, so several are tried in turn. If a file is missing a required column, the error names it.
- **Cleaning:**
  1. *Keep only songs.* Podcast episodes have URIs starting with `spotify:episode:` and no meaningful audio features.
  2. *Drop songs Spotify couldn't analyze.* That means rows with missing features, or with Danceability and Tempo both 0, which is how Spotify marks a failed analysis. These aren't real sound profiles.
  3. *One row per song.* The same song can appear in several files. Duplicates would count twice and pull clusters toward themselves, so each `Track URI` is kept once.

**For the recommender:** `Track URI` is the **unique ID** of each song in the output. Use it, not the track name, to look songs up, since different songs can share a name.

In [2]:
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]
REQUIRED_COLUMNS = ["Track URI", "Track Name", "Artist Name(s)", "Tempo"] + FEATURES


def read_export(path):
    # Read one CSV, trying each encoding until one works
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            continue
    raise ValueError(f"Could not read {path.name} with any of {ENCODINGS}")


frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    frame = read_export(path)
    missing = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")
    frames.append(frame)

if not frames:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")
data = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(data)} rows from {len(frames)} file(s)")

Loaded 3197 rows from 2 file(s)


In [3]:
is_song = data["Track URI"].str.startswith("spotify:track:")
analyzed = data[FEATURES].notna().all(axis=1) & ~(data["Danceability"].eq(0) & data["Tempo"].eq(0))

songs = data[is_song & analyzed].drop_duplicates("Track URI").reset_index(drop=True)

print(f"Removed {(~is_song).sum()} non-songs and {(is_song & ~analyzed).sum()} unanalyzed songs")
print(f"{len(songs)} unique songs to cluster")

Removed 1 non-songs and 1 unanalyzed songs
3156 unique songs to cluster


## Step 2 — Prepare the audio features on a fixed scale

**What was done:** the 7 audio features in `FEATURES` were converted to a common 0–1 scale with a **fixed formula**, so each one counts fairly when measuring how similar two songs are.

**Why these features:** they describe how a song *sounds*.

| Feature | Raw range | Meaning |
|---|---|---|
| Danceability | 0–1 | how suitable for dancing (rhythm, beat strength) |
| Energy | 0–1 | intensity and activity |
| Loudness | about −60 to 0 dB | overall volume |
| Speechiness | 0–1 | spoken words (high for rap, podcasts) |
| Acousticness | 0–1 | acoustic vs. electronic/amplified |
| Instrumentalness | 0–1 | likelihood of no vocals |
| Valence | 0–1 | musical positivity (happy vs. sad/dark) |

Left out: **Tempo** (Spotify often reports half or double the real tempo), **Key / Mode / Time Signature** (categories, not sound qualities), **Liveness** (live vs. studio recording, not style) and **Popularity / Duration** (not about sound).

**How:**

1. **Loudness** is the only feature not already on 0–1, so it's mapped from Spotify's fixed range `LOUDNESS_RANGE` (−60 to 0 dB) onto 0–1. The rare values outside that range are clipped.
2. **Speechiness, Acousticness and Instrumentalness** (`SQRT_FEATURES`) are bunched near 0 for most songs. Taking the **square root** spreads out the small values (0.04 → 0.2, 0.25 → 0.5) so that differences among typical songs count, while keeping the 0–1 range.

**Why a fixed formula instead of `StandardScaler`:** `StandardScaler` rescales each feature using *this dataset's* mean and standard deviation, so "high energy" would mean "high compared with these particular songs." Adding songs would shift every value, and songs scaled in two different datasets couldn't be compared. A fixed formula gives a song **the same prepared values in any dataset**. That's what lets clusters and similarity carry over to new data.

**For the recommender:** the saved `(prepared)` columns are these values. To prepare a new song, apply `prepare_features` (below) to its raw features. It's a formula, not a fitted model, so nothing needs to be loaded for this step.

In [4]:
def prepare_features(table):
    # Convert raw Spotify features to a fixed 0-1 scale; the same song always gets the same values
    prepared = table[FEATURES].astype(float)
    low, high = LOUDNESS_RANGE
    prepared["Loudness"] = (prepared["Loudness"] - low) / (high - low)
    prepared = prepared.clip(0, 1)
    prepared[SQRT_FEATURES] = np.sqrt(prepared[SQRT_FEATURES])
    return prepared


X = prepare_features(songs)

PREPARED_FEATURES = [feature + " (prepared)" for feature in FEATURES]
songs[PREPARED_FEATURES] = X.to_numpy()

# Every prepared feature now runs from 0 to 1
X.describe().round(2)

,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence
count,3156.00,3156.00,3156.00,3156.00,3156.00,3156.00,3156.00
mean,0.57,0.65,0.88,0.29,0.39,0.06,0.47
std,0.17,0.21,0.06,0.13,0.30,0.18,0.23
min,0.09,0.00,0.23,0.15,0.00,0.00,0.00
25%,0.46,0.50,0.86,0.19,0.11,0.00,0.29
50%,0.58,0.67,0.89,0.23,0.33,0.00,0.45
75%,0.70,0.83,0.92,0.34,0.63,0.02,0.64
max,0.98,1.00,1.00,0.90,1.00,1.00,0.98


## Step 3 — Choose the number of clusters

**What was done:** k-means was run for every number of clusters in `K_OPTIONS`, and three scores were compared to pick `N_CLUSTERS`.

**How:**

- **Inertia** (elbow method): the total squared distance from each song to its cluster's center. It always drops as k grows; the point where the drop levels off (the "elbow") is a good trade-off.
- **Silhouette**: for each song, how much closer it is to its own cluster than to the nearest other cluster, averaged over all songs. It runs from −1 to 1, and higher means better-separated clusters.
- **Stability** (the test for generalization): split the songs randomly into two halves, fit k-means on each half separately, then label *every* song with both models and measure how much the two labelings agree. Agreement is measured with the **adjusted Rand index**: 1 means identical groupings, 0 means no better than chance, and the cluster numbers themselves don't matter. This is repeated `STABILITY_REPEATS` times and averaged.

A k that gives **high stability** produces the same clusters whichever songs it learns from, which is the best evidence available that the clusters describe general sound patterns and would hold up on a new dataset. Values of k larger than the songs available in half the data are skipped, so small datasets don't crash.

In [5]:
def split_half_agreement(X, k):
    # Fit k-means on two random halves, label every song with both, and average their agreement (adjusted Rand index)
    rng = np.random.default_rng(RANDOM_STATE)
    agreements = []
    for repeat in range(STABILITY_REPEATS):
        shuffled = rng.permutation(len(X))
        half_a, half_b = shuffled[: len(X) // 2], shuffled[len(X) // 2:]
        model_a = KMeans(n_clusters=k, n_init=10, random_state=repeat).fit(X.iloc[half_a])
        model_b = KMeans(n_clusters=k, n_init=10, random_state=repeat).fit(X.iloc[half_b])
        agreements.append(adjusted_rand_score(model_a.predict(X), model_b.predict(X)))
    return np.mean(agreements)


scores = []
for k in K_OPTIONS:
    if k >= len(X) // 2:  # not enough songs in each half for this many clusters
        break
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    scores.append({"k": k,
                   "inertia": model.inertia_,
                   "silhouette": silhouette_score(X, model.labels_),
                   "stability": split_half_agreement(X, k)})
scores = pd.DataFrame(scores)

px.line(scores, x="k", y="inertia", markers=True, title="Elbow method").show()
px.line(scores.melt(id_vars="k", value_vars=["silhouette", "stability"], var_name="score"),
        x="k", y="value", color="score", markers=True,
        title="Separation (silhouette) and stability across random halves").show()
scores.round(3)

,k,inertia,silhouette,stability
0,2,564.785,0.302,0.906
1,3,468.775,0.250,0.826
2,4,409.569,0.263,0.451
3,5,358.275,0.231,0.471
4,6,315.978,0.238,0.874
5,7,295.506,0.245,0.774
6,8,277.246,0.241,0.705
7,9,263.749,0.207,0.552
8,10,249.999,0.201,0.563
9,11,240.680,0.197,0.537


**In the current data:**

- **Silhouette is low for every k** (about 0.2–0.3), which is normal for music. Songs don't form tight, separate islands; sounds blend into each other.
- **Stability separates the options.** k = 2 is the most stable (0.91), but two clusters (roughly loud vs. soft) are too coarse to be useful. k = 4 and 5 are unstable (about 0.45): their clusters change depending on which songs they learn from. **k = 6 is the most stable useful choice (0.87)**, and stability drops again from k = 7 (0.77) onward.
- So we use **k = 6**: its clusters come out nearly the same whichever half of the songs they're learned from.

**With new data:** re-check the table and set `N_CLUSTERS` in Settings to a k with high stability (above about 0.8) that still gives enough clusters to be useful. Prefer the smaller k when two are close.

**For the recommender:** because the clusters overlap, songs near the edge of a cluster can sound a lot like songs in the neighbouring cluster. Searching only within one cluster will miss some good matches.

## Step 4 — Cluster the songs and describe each cluster

**What was done:** every song was assigned to one of `N_CLUSTERS` clusters, and each cluster got a plain-words **sound description**.

**How:**

1. **k-means** places `N_CLUSTERS` centers among the prepared features and assigns each song to its **nearest center** by straight-line (Euclidean) distance. It then moves each center to the average of its songs and repeats until nothing changes. `n_init=10` runs this from 10 random starting points and keeps the best result, and `RANDOM_STATE` makes it repeatable.
2. **Describing each cluster.** Each cluster is summarized by its **median** song in raw Spotify units. Medians are used because they aren't pulled around by a few extreme songs. `describe_sound` turns that profile into words using **fixed thresholds**, so a label means the same thing in any dataset. For example, "high energy" always means a median Energy above 0.7, never "high compared with these songs." The Speechiness (0.33) and Instrumentalness (0.5) cut-offs follow Spotify's own descriptions of those features.

**For the recommender:**

- `model.predict(prepare_features(table))` assigns new songs to clusters. Prepare them with `prepare_features` first (Step 2).
- `model.cluster_centers_` holds the centers on the prepared 0–1 scale.
- **Cluster numbers are just IDs.** They can come out shuffled after re-running with new data or a different scikit-learn version. Use the `Sound` description, not the number, to know what a cluster sounds like.

In [6]:
model = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE).fit(X)
songs["Cluster"] = model.labels_


def describe_sound(profile):
    # Plain-words label from a cluster's median raw features, using fixed thresholds
    words = []
    if profile["Instrumentalness"] > 0.5:
        words.append("instrumental")
    if profile["Speechiness"] > 0.33:
        words.append("spoken / rap")
    words.append("high energy" if profile["Energy"] > 0.7 else "low energy" if profile["Energy"] < 0.45 else "medium energy")
    words.append("upbeat" if profile["Valence"] > 0.6 else "dark" if profile["Valence"] < 0.4 else "neutral mood")
    if profile["Acousticness"] > 0.5:
        words.append("acoustic")
    if profile["Danceability"] > 0.65:
        words.append("danceable")
    return ", ".join(words)


# Each cluster's median song in raw units, its description, and its size
cluster_profile = songs.groupby("Cluster")[FEATURES].median().round(2)
cluster_profile["Sound"] = cluster_profile.apply(describe_sound, axis=1)
cluster_profile["Songs"] = songs["Cluster"].value_counts()
songs["Sound"] = songs["Cluster"].map(cluster_profile["Sound"])
cluster_profile

,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Sound,Songs
Cluster,,,,,,,,,
0,0.51,0.52,-10.43,0.05,0.50,0.75,0.30,"instrumental, medium energy, dark",131
1,0.43,0.88,-4.75,0.05,0.00,0.00,0.37,"high energy, dark",670
2,0.63,0.57,-7.20,0.06,0.11,0.00,0.29,"medium energy, dark",608
3,0.51,0.37,-9.25,0.04,0.70,0.00,0.32,"low energy, dark, acoustic",516
4,0.70,0.62,-7.06,0.12,0.34,0.00,0.65,"medium energy, upbeat, danceable",591
5,0.63,0.80,-5.38,0.05,0.03,0.00,0.71,"high energy, upbeat",640


In [7]:
# Four example songs from each cluster
for cluster, group in songs.groupby("Cluster"):
    examples = group.sample(min(4, len(group)), random_state=1)
    print(f"Cluster {cluster} ({group['Sound'].iloc[0]}):")
    print("   " + " | ".join(examples["Track Name"] + " — " + examples["Artist Name(s)"]))

Cluster 0 (instrumental, medium energy, dark):
   Seasons - club mix — Sam Austins | Beethoven's 5 Secrets — The Piano Guys;Lyceum Philharmonic at American Heritage School;Julie Ann Nelson;John Nelson | Forever Falling — Alter Bridge | Chamber Of Reflection — Mac DeMarco
Cluster 1 (high energy, dark):
   I Found Away feat. Douglas P — Alkaline Trio | Circus for a Psycho — Skillet | Find The Real — Alter Bridge | Vaudeville Villain — Viktor Vaughn;MF DOOM
Cluster 2 (medium energy, dark):
   Stay Gold — Bryson Tiller | If I Get High — Nothing But Thieves | EARFQUAKE — Tyler, The Creator | Broken — Seether
Cluster 3 (low energy, dark, acoustic):
   A Thousand Miles — Boyce Avenue;Alex Goot | Want You — Clara La San | Self Control — Frank Ocean | She Banking on Me Pt. 2 — shelovesavoidd
Cluster 4 (medium energy, upbeat, danceable):
   Would've, Could've, Should've — Taylor Swift | Welcome To Brixton — SR | GYALIS — Capella Grey | Intergalactic Janet — Ley Soul
Cluster 5 (high energy, upbea

## Step 5 — Save the results for a future recommender

**What was done:** the clustered songs and the fitted k-means model were saved to `data/processed/`. That folder is skipped when loading in Step 1, because Step 1 only reads CSVs directly inside `data/`. Re-running the notebook overwrites both files.

**`songs_with_clusters.csv`**: one row per unique song.

| Column | Meaning |
|---|---|
| `Track URI` | Spotify's unique song ID; use it as the key |
| `Track Name` | song title (not unique) |
| `Artist Name(s)` | artists, separated by `;` |
| `Cluster` | cluster ID from Step 4 (an ID only; see `Sound` for meaning) |
| `Sound` | plain-words description of the cluster's typical sound, e.g. `high energy, upbeat` |
| the 7 features in `FEATURES` | raw Spotify values (Step 2) |
| the same 7 with ` (prepared)` | fixed-scale 0–1 values used for clustering; distance between songs is measured on these |

**`cluster_model.joblib`**: the fitted k-means model from Step 4. Load it with `joblib.load(...)` and pass it songs prepared with `prepare_features`. It was built with the scikit-learn version printed below; loading it with a very different version can fail or give a warning, and re-running this notebook rebuilds it.

In [8]:
OUTPUT_DIR.mkdir(exist_ok=True)
columns_to_save = ["Track URI", "Track Name", "Artist Name(s)", "Cluster", "Sound"] + FEATURES + PREPARED_FEATURES
songs[columns_to_save].to_csv(OUTPUT_DIR / "songs_with_clusters.csv", index=False)
joblib.dump(model, OUTPUT_DIR / "cluster_model.joblib")

print(f"Saved {len(songs)} songs and the model to {OUTPUT_DIR.resolve()}")
print("scikit-learn version:", sklearn.__version__)

Saved 3156 songs and the model to C:\Users\diall\OneDrive\ADS_DSCI D501 Projects\Spotify-Playlist-Data-Analysis\data\processed
scikit-learn version: 1.9.1


## Summary and notes for building the recommender

**What this notebook does:** it groups songs into clusters by sound alone, using 7 Spotify audio features prepared on a fixed scale. In the current data, 6 clusters are the most stable useful choice (Step 3). They range from high energy and dark, through upbeat and danceable, to low energy and acoustic, plus a small instrumental group (Step 4).

**Why the clusters carry over to new datasets:**

- The feature preparation is a fixed formula (Step 2), so a song's values never depend on which other songs are in the dataset.
- The number of clusters was chosen for **stability**: clusters learned from different random halves of the songs agree closely (Step 3).
- The `Sound` labels use fixed thresholds (Step 4), so "high energy" or "acoustic" means the same thing everywhere.

**What a recommender builder should know:**

1. **Similarity = distance on the `(prepared)` columns.** Two songs are "similar" when their prepared values are close (e.g. Euclidean distance). That's the same measure k-means used, and because the scale is fixed, distances are comparable across datasets.
2. **Clusters can narrow the search.** Comparing a song only with songs in its own `Cluster` is faster and keeps the overall sound consistent. Clusters overlap, though (Step 3), so close matches in a neighbouring cluster will be missed.
3. **New songs** need the 7 raw features. Run them through `prepare_features`, then `cluster_model.joblib` assigns a cluster (Step 4).
4. **Sound only.** These clusters know nothing about genre, lyrics or artist, so similar-sounding songs can come from any genre.
5. **The cluster centers were learned from this data's mix of music.** If a new dataset contains sounds this one barely has (say, mostly EDM or classical), those songs still get assigned to the nearest existing cluster, but a better grouping may exist. Re-run this notebook on the combined data and re-check the stability table.
6. **Cluster IDs can change** when the notebook is re-run, so join results on `Track URI` and read clusters by their `Sound` description.